# Tree accuracy: does the tree agree with the database taxonomy?

Executed by the `tree_accuracy` rule once per tree variant, after the tree, PhyloRank and
`assign_taxonomy` have run. The question: **how well do the database clade names (phylum ... genus)
match the tree, and where they do not, is it a tree problem or a label problem?**

| label | meaning |
|---|---|
| `all` / `derep` | every genome, or only the dereplicated representatives |
| `single` / `multi` | Complete BUSCO genes only, or also the top copy of Duplicated genes |
| `subset ...` | quick-check subset - do not compare its numbers with a full run |

Sections: (1) clade fragmentation, (2) support of the splits that cause it, (3) tips that look misplaced,
(4) RED statistics. All monophyly numbers are **independent of where the tree is rooted**
(see `scripts/tree_accuracy_lib.py`).

## 0. Setup

Parameters come from the JSON file named by `NB_PARAMS`, written by the Snakemake rule.

In [ ]:
import csv
import json
import os
import sys
from pathlib import Path

import numpy as np
import polars as pl
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
%matplotlib inline
from Bio import Phylo

In [ ]:
PARAMS = json.load(open(os.environ["NB_PARAMS"]))
sys.path.insert(0, PARAMS["scripts_dir"])
import tree_accuracy_lib as lib

OUT = Path(PARAMS["out_dir"])
OUT.mkdir(parents=True, exist_ok=True)

print("Tree variant:", PARAMS["label"])
print("Tree        :", PARAMS["tree"])

In [ ]:
def show_all(df, width=200):
    '''
    Prints an entire polars dataframe in the notebook output.

    Parameters
    ----------
    df : pl.DataFrame
        The dataframe to be printed.
    width : int, optional
        The width (in characters) of the printed dataframe.
        Defaults to 200.
    '''
    with pl.Config() as cfg:
        cfg.set_tbl_cols(-1)
        cfg.set_tbl_rows(-1)
        cfg.set_tbl_width_chars(width)
        cfg.set_fmt_str_lengths(width)
        print(df)

In [ ]:
tree = Phylo.read(PARAMS["tree"], "newick")
taxonomy = lib.read_taxonomy(PARAMS["taxonomy"])  # {genome: {rank: name}} from the database lineages

n_tips = tree.count_terminals()
n_labelled = sum(1 for tip in tree.get_terminals() if tip.name in taxonomy)
print(f"{n_tips} tips, {n_labelled} with a database taxonomy ({n_tips - n_labelled} unlabelled, e.g. MAGs)")

## 1. Clade fragmentation

If a database name (say an order) is monophyletic, all its tips form **one piece**: one part of the tree
that contains tips of that name and no labelled tips of any other name. Tips of the same name that sit
elsewhere form **extra pieces**. Unlabelled tips (e.g. MAGs) are ignored, so they never count against a name.

Columns: `names` = distinct names at the rank; `fragmented` = names with more than one piece;
`extra pieces` = pieces beyond the first; `tips in frag.` = tips of fragmented names.

**How to read it.**

* A few extra pieces at deep ranks (phylum, class) are usually real: NCBI taxa like Pichiomycetes or
  Bigyra are known to be non-monophyletic.
* Fragmentation that grows toward the tips (many extra pieces at order and family) is more likely wrong
  labels, contaminated genomes or an unstable tree.
* Extra pieces of size 1 are single genomes sitting outside their group - look at them in section 3.

In [ ]:
pieces = {rank: lib.pieces_unrooted(tree, taxonomy, rank) for rank in lib.RANKS}
summary, names = lib.fragmentation(tree, taxonomy, pieces_by_rank=pieces)

fragmentation = pl.DataFrame(summary)
show_all(fragmentation)

per_name = pl.DataFrame([
    {"rank": n["rank"], "name": n["name"], "tips": n["tips"], "pieces": n["pieces"],
     "extra_piece_sizes": ",".join(map(str, n["extra_sizes"]))}
    for n in names
])
per_name.write_csv(OUT / "fragmentation.tsv", separator="\t")

In [ ]:
# The most fragmented names at the deeper ranks
for rank in ("phylum", "class", "order"):
    worst = per_name.filter((pl.col("rank") == rank) & (pl.col("pieces") > 1)).sort("pieces", descending=True).head(8)
    print(f"Most fragmented {rank}s:")
    show_all(worst.select("name", "tips", "pieces", "extra_piece_sizes"))

## 2. Are the splits that fragment a clade supported?

Each extra piece hangs below a branch. Its **support** is FastTree's SH-like value on that branch
(0-1). This separates two situations:

* **high support (>= 0.95)** - the data confidently place those tips elsewhere. The disagreement with the
  database is real (a wrong label, a contaminated genome, or biology), and the tree is not the problem.
* **low support** - the placement is uncertain and the tree may simply be wrong there.

SH-like values are generous, so treat high support as "not obviously noise", not as proof.

In [ ]:
support_rows = []
for rank in lib.RANKS:
    for name, size, support in lib.extra_piece_supports(pieces[rank]):
        support_rows.append({"rank": rank, "name": name, "piece_tips": size, "support": support})
extra_support = pl.DataFrame(support_rows, schema={"rank": pl.Utf8, "name": pl.Utf8,
                                                    "piece_tips": pl.Int64, "support": pl.Float64})
extra_support.write_csv(OUT / "extra_piece_support.tsv", separator="\t")

with_support = extra_support.drop_nulls("support")
if with_support.height:
    show_all(with_support.select(
        pl.len().alias("extra_pieces"),
        pl.col("support").median().round(3).alias("median_support"),
        (pl.col("support") < 0.8).mean().round(3).alias("share_below_0.8"),
        (pl.col("support") < 0.9).mean().round(3).alias("share_below_0.9"),
        (pl.col("support") >= 0.95).mean().round(3).alias("share_at_least_0.95"),
    ))
else:
    print("No support values in this tree.")

In [ ]:
if with_support.height:
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
    axes[0].hist(with_support["support"], bins=20)
    axes[0].set_title("support of the branch above each extra piece")
    axes[1].scatter(with_support["piece_tips"], with_support["support"], s=6)
    axes[1].set_xscale("log")
    axes[1].set_xlabel("piece size (tips)")
    axes[1].set_ylabel("support")
    plt.show()

## 3. Tips that look misplaced

A tip is flagged at a rank when its nearest labelled neighbours (at least 8) **agree at >= 90% on a different
name**. `sister = True` means that other name belongs to the same parent clade (for example two orders
of one class), which is a milder disagreement and often just a classification difference; the table keeps
both, but the summary below counts only `sister = False`.

We then compare the flagged genomes with all genomes on three things that would point at the genome
rather than the label: BUSCO completeness, BUSCO duplication (a high fraction can mean contamination or a
mixed assembly) and whether their database taxonomy is only `partial`.

**How to read it.** If flagged genomes look like everyone else, the cause is most likely the label (or real
biology). If they are less complete, more duplicated or more often `partial`, look at the genomes first.

In [ ]:
# tax_status (complete / partial / unresolved) of every genome, from the metapackage taxonomy mapping
mapping = pl.read_csv(PARAMS["taxonomy_mapping"], separator="\t").select("acc", "tax_status")
tax_status = dict(zip(mapping["acc"], mapping["tax_status"]))

# BUSCO full table of every genome
busco_tables = {}
for entry in PARAMS["busco_manifests"]:
    manifest, base = Path(entry["manifest"]), Path(entry["base"])
    if not manifest.exists():
        continue
    qc = pl.read_csv(manifest, separator="\t")
    for accession, result in zip(qc["genome_acc"], qc["busco_result"]):
        busco_tables[accession] = Path(result) if Path(result).is_absolute() else base / result
print(f"{len(busco_tables)} BUSCO tables, {len(tax_status)} taxonomy entries")

In [ ]:
flagged = {rank: lib.misplaced_tips(tree, taxonomy, rank) for rank in ("phylum", "class", "order", "family")}

flag_counts = pl.DataFrame([
    {"rank": rank, "flagged": len(rows),
     "sister_group_swaps": sum(r["sister"] for r in rows),
     "non_sister": sum(not r["sister"] for r in rows)}
    for rank, rows in flagged.items()
])
show_all(flag_counts)

misplaced = pl.DataFrame(
    [{"rank": rank, **row} for rank, rows in flagged.items() for row in rows],
    schema={"rank": pl.Utf8, "tip": pl.Utf8, "label": pl.Utf8, "neighbours_say": pl.Utf8,
            "agreement": pl.Float64, "n_neighbours": pl.Int64, "sister": pl.Boolean},
)
misplaced.write_csv(OUT / "misplaced_tips.tsv", separator="\t")

In [ ]:
def genome_summary(accessions, name):
    '''
    Summarises BUSCO completeness, duplication and taxonomy status of a set of genomes.

    Parameters
    ----------
    accessions : iterable of str
        Genome accessions (tree tip names).
    name : str
        Label for the returned row.

    Returns
    -------
    dict
        One summary row (genomes without a BUSCO table are skipped for the BUSCO columns).
    '''
    accessions = list(accessions)
    stats = []
    for accession in accessions:
        path = busco_tables.get(accession)
        if path is not None and Path(path).exists():
            counts = lib.busco_status_counts(path)
            total = sum(counts.values()) or 1
            stats.append((counts.get("Complete", 0), counts.get("Duplicated", 0), counts.get("Duplicated", 0) / total))
    partial = [tax_status.get(accession) == "partial" for accession in accessions]
    return {
        "group": name, "genomes": len(accessions), "with_busco": len(stats),
        "median_complete": float(np.median([s[0] for s in stats])) if stats else None,
        "median_duplicated": float(np.median([s[1] for s in stats])) if stats else None,
        "median_dup_fraction": round(float(np.median([s[2] for s in stats])), 3) if stats else None,
        "share_taxonomy_partial": round(float(np.mean(partial)), 3) if partial else None,
    }


non_sister_tips = set(misplaced.filter(~pl.col("sister") & pl.col("rank").is_in(["order", "family"]))["tip"])
all_tips = [tip.name for tip in tree.get_terminals()]
show_all(pl.DataFrame([
    genome_summary(sorted(non_sister_tips), "non-sister misplaced (order/family)"),
    genome_summary(all_tips, "all tips"),
]))

In [ ]:
print("Non-sister order-level examples:")
show_all(misplaced.filter((pl.col("rank") == "order") & ~pl.col("sister")).sort("agreement", descending=True)
         .head(15).select("tip", "label", "neighbours_say", "agreement", "n_neighbours"))

## 4. RED (relative evolutionary divergence)

PhyloRank gives every node a RED value between 0 (root) and 1 (tips); `assign_taxonomy` uses RED cutoffs
per rank to decide how deep a database name may be inherited by the tips under a node. The cutoffs are the
median RED of the existing names at each rank on **this** tree.

**Why check it.**

* *Spread*: if the RED values of, say, orders overlap a lot with those of families, a single cutoff cannot
  separate the ranks. `wrong side` below is the share of names at a rank whose RED falls on the wrong side of
  the neighbouring rank's median. Small is good; 10-20% means RED only roughly follows the database ranks
  (expected: NCBI ranks are not equal in age across the tree of eukaryotes).
* *Monotonicity*: PhyloRank's RED is a median over several rootings, so a child can end up below its
  parent, which also produces negative branch lengths in the RED tree. `assign_taxonomy` ignores those
  lengths and raises each node's RED to at least its parent's. A handful of nodes is fine; many would mean the
  tree is poorly rooted or unstable near the root.

In [ ]:
red = lib.red_by_rank(PARAMS["red_tsv"], PARAMS["red_dict"])  # {rank: (cutoff, [RED of each named taxon])}

rows = []
for rank in ("phylum", "class", "order", "family", "genus", "species"):
    cutoff, values = red.get(rank, (None, []))
    if not values:
        continue
    values = np.asarray(values)
    rows.append({"rank": rank, "cutoff": round(cutoff, 3), "taxa": len(values),
                 "q10": round(float(np.quantile(values, 0.1)), 3),
                 "median": round(float(np.median(values)), 3),
                 "q90": round(float(np.quantile(values, 0.9)), 3)})
red_table = pl.DataFrame(rows)
show_all(red_table)
red_table.write_csv(OUT / "red_summary.tsv", separator="\t")

In [ ]:
overlap = lib.rank_overlap(red)
show_all(pl.DataFrame([
    {"rank": rank, "below_parent_rank_median": round(below, 3), "above_child_rank_median": round(above, 3)}
    for rank, (below, above) in overlap.items()
]))

ranks_with_data = [rank for rank in ("phylum", "class", "order", "family", "genus") if red.get(rank) and red[rank][1]]
fig, axis = plt.subplots(figsize=(8, 3.5))
axis.boxplot([red[rank][1] for rank in ranks_with_data])
axis.set_xticks(range(1, len(ranks_with_data) + 1))
axis.set_xticklabels(ranks_with_data)
axis.plot(range(1, len(ranks_with_data) + 1), [red[rank][0] for rank in ranks_with_data],
          "r_", markersize=30, label="cutoff")
axis.set_ylabel("RED of each named taxon")
axis.legend()
plt.show()

In [ ]:
red_tree = Phylo.read(PARAMS["red_tree"], "newick")
n_nodes, n_below_parent, n_negative, most_negative = lib.red_monotonicity(red_tree)
print(f"PhyloRank RED tree: {n_nodes} internal nodes with RED, {n_below_parent} below their parent, "
      f"{n_negative} negative branch lengths (most negative {most_negative:.4f})")

## 5. Summary

Tables written next to this notebook: `fragmentation.tsv`, `extra_piece_support.tsv`,
`misplaced_tips.tsv`, `red_summary.tsv`. Compare variants in the `tree_comparison` notebook.

In [ ]:
print(f"Variant: {PARAMS['label']}   tips: {n_tips} ({n_labelled} labelled)")
for row in fragmentation.iter_rows(named=True):
    print(f"  {row['rank']:7} {row['fragmented_names']:4d}/{row['names']:<5d} names fragmented, "
          f"{row['extra_pieces']:4d} extra pieces")
if with_support.height:
    print(f"Extra pieces with support >= 0.95: {(with_support['support'] >= 0.95).mean():.0%}")
print(f"Non-sister misplaced tips (order/family): {len(non_sister_tips)}")
print(f"RED: {n_below_parent} nodes below their parent, {n_negative} negative branches")